In [19]:
#Célula 1: Instalação de dependências
#%pip install requests tenacity psycopg2-binary python-dotenv tqdm --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\fabri\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [38]:
#Célula 2: imports e configuração de pastas
import os, json, time, logging
from pathlib import Path
from datetime import datetime

import requests
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
import psycopg2
from psycopg2.extras import Json, execute_values
from tqdm.notebook import tqdm
from dotenv import load_dotenv

import pandas as pd

load_dotenv()

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("jurisdata")

DATA_DIR = Path("../data")
RAW_DIR  = DATA_DIR / "raw"
CHK_DIR  = DATA_DIR / "checkpoints"
for d in [RAW_DIR, CHK_DIR]:
    d.mkdir(parents=True, exist_ok=True)

log.info("Pastas OK: %s", DATA_DIR.resolve())

2026-05-17 02:50:45,436 [INFO] Pastas OK: C:\Users\fabri\Desktop\Portfolio\jurisdata\data


In [21]:
#Célula 3: constantes da API
DATAJUD_API_KEY = os.getenv("DATAJUD_API_KEY")
ENDPOINT_TRT21  = "https://api-publica.datajud.cnj.jus.br/api_publica_trt21/_search"

HEADERS = {
    "Authorization": f"APIKey {DATAJUD_API_KEY}",
    "Content-Type": "application/json",
}

CLASSE_CODIGO    = 985
DATE_FROM        = "20180101000000"
DATE_TO          = "20241231000000"
PAGE_SIZE        = 1000
MAX_PAGES        = None

CHECKPOINT_FILE  = CHK_DIR / f"checkpoint_trt21_{DATE_FROM[:4]}_{DATE_TO[:4]}.json"

log.info("Constantes carregadas. MAX_PAGES=%s", MAX_PAGES)

2026-05-17 02:40:06,690 [INFO] Constantes carregadas. MAX_PAGES=None


In [22]:
#Célula 4: conexão e criação dos schemas/tabelas no PostgreSQL
PG_CONFIG = {
    "host":     os.getenv("PG_HOST", "localhost"),
    "port":     int(os.getenv("PG_PORT", "5432")),
    "dbname":   os.getenv("PG_DBNAME", "jurisdata"),
    "user":     os.getenv("PG_USER", "postgres"),
    "password": os.getenv("PG_PASSWORD", ""),
}

def get_conn():
    return psycopg2.connect(**PG_CONFIG)

# Testa a conexão
with get_conn() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version();")
        log.info("PostgreSQL: %s", cur.fetchone()[0][:50])

2026-05-17 02:40:06,969 [INFO] PostgreSQL: PostgreSQL 16.13, compiled by Visual C++ build 194


In [23]:
#Célula 5: DDL: criação dos schemas e tabelas
DDL = """
CREATE SCHEMA IF NOT EXISTS staging;
CREATE SCHEMA IF NOT EXISTS analytics;
CREATE SCHEMA IF NOT EXISTS ml;
CREATE SCHEMA IF NOT EXISTS llm;

CREATE TABLE IF NOT EXISTS staging.decisoes_raw (
    id          BIGSERIAL PRIMARY KEY,
    process_id  TEXT UNIQUE NOT NULL,
    tribunal    TEXT NOT NULL DEFAULT 'TRT21',
    raw_json    JSONB NOT NULL,
    ingested_at TIMESTAMPTZ DEFAULT NOW()
);
CREATE INDEX IF NOT EXISTS idx_raw_json     ON staging.decisoes_raw USING GIN (raw_json);
CREATE INDEX IF NOT EXISTS idx_raw_ingested ON staging.decisoes_raw (ingested_at DESC);

CREATE TABLE IF NOT EXISTS analytics.processos (
    id                    BIGSERIAL PRIMARY KEY,
    process_id            TEXT UNIQUE NOT NULL,
    numero_cnj            TEXT,
    tribunal              TEXT,
    classe_codigo         INTEGER,
    classe_nome           TEXT,
    data_ajuizamento      DATE,
    data_ultima_atualizacao DATE,
    assunto_principal     TEXT,
    assunto_codigo        INTEGER,
    grau                  TEXT,
    orgao_julgador        TEXT,
    formato               TEXT,
    tempo_tramitacao_dias INTEGER,
    created_at            TIMESTAMPTZ DEFAULT NOW()
);
CREATE INDEX IF NOT EXISTS idx_proc_classe   ON analytics.processos (classe_codigo);
CREATE INDEX IF NOT EXISTS idx_proc_assunto  ON analytics.processos (assunto_codigo);
CREATE INDEX IF NOT EXISTS idx_proc_data     ON analytics.processos (data_ajuizamento);
"""

with get_conn() as conn:
    with conn.cursor() as cur:
        cur.execute(DDL)
    conn.commit()

log.info("Schemas e tabelas criados: staging, analytics, ml, llm")

2026-05-17 02:40:07,172 [INFO] Schemas e tabelas criados: staging, analytics, ml, llm


In [34]:
#Célula 6:  funções de query para a API
def build_query(search_after=None):
    q = {
        "query": {
            "bool": {
                "must": [
                    {"term": {"classe.codigo": str(CLASSE_CODIGO)}},
                    {"range": {"dataAjuizamento": {"gte": DATE_FROM, "lte": DATE_TO}}},
                ]
            }
        },
        "size": PAGE_SIZE,
        "sort": [
            {"dataAjuizamento": {"order": "desc"}},
            {"@timestamp": {"order": "asc"}},
        ],
    }
    if search_after:
        q["search_after"] = search_after
    return q

@retry(
    retry=retry_if_exception_type((requests.Timeout, requests.ConnectionError)),
    wait=wait_exponential(multiplier=2, min=4, max=60),
    stop=stop_after_attempt(5),
    reraise=True,
)
def fetch_page(search_after=None):
    r = requests.post(ENDPOINT_TRT21, headers=HEADERS, json=build_query(search_after), timeout=30)
    r.raise_for_status()
    return r.json()

def count_total():
    r = requests.post(ENDPOINT_TRT21, headers=HEADERS,
                      json={"query": build_query()["query"], "size": 0}, timeout=30)
    r.raise_for_status()
    return r.json()["hits"]["total"]["value"]


total = count_total()
log.info("Total disponível no TRT-21 (período configurado): %s", f"{total:,}")

2026-05-17 02:42:39,724 [INFO] Total disponível no TRT-21 (período configurado): 10,000


In [36]:
#Célula 7: funções de checkpoint e inserção no banco
def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE) as f:
            chk = json.load(f)
        log.info("Checkpoint encontrado: %s registros, retomando de search_after=%s",
                 chk["total_ingested"], chk["last_search_after"])
        return chk
    return {"total_ingested": 0, "last_search_after": None, "pages_fetched": 0}


def save_checkpoint(total_ingested, last_search_after, pages_fetched):
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({
            "total_ingested": total_ingested,
            "last_search_after": last_search_after,
            "pages_fetched": pages_fetched,
            "updated_at": datetime.utcnow().isoformat(),
        }, f, indent=2)


def upsert_batch(conn, batch):
    if not batch:
        return 0
    rows = [(doc["_id"], "TRT21", Json(doc["_source"])) for doc in batch]
    sql = """
        INSERT INTO staging.decisoes_raw (process_id, tribunal, raw_json)
        VALUES %s
        ON CONFLICT (process_id) DO NOTHING
    """
    with conn.cursor() as cur:
        execute_values(cur, sql, rows, page_size=500)
        inserted = cur.rowcount
    conn.commit()
    return inserted

In [37]:
#Célula 8: pipeline principal de ingestão
def run_ingestion():
    state          = load_checkpoint()
    search_after   = state["last_search_after"]
    total_ingested = state["total_ingested"]
    pages_fetched  = state["pages_fetched"]

    log.info("Iniciando. Já ingeridos: %s", f"{total_ingested:,}")
    conn = get_conn()
    try:
        while True:
            if MAX_PAGES and pages_fetched >= MAX_PAGES:
                log.info("MAX_PAGES=%s atingido.", MAX_PAGES)
                break

            try:
                data = fetch_page(search_after)
            except requests.HTTPError as e:
                log.error("Erro HTTP: %s — checkpoint salvo, reexecute para retomar.", e)
                break

            hits = data.get("hits", {}).get("hits", [])
            if not hits:
                log.info("Sem mais resultados. Ingestão concluída.")
                break

            # Salva raw em disco
            raw_path = RAW_DIR / f"trt21_page_{pages_fetched + 1:05d}.json"
            with open(raw_path, "w", encoding="utf-8") as f:
                json.dump(hits, f, ensure_ascii=False)

            inserted       = upsert_batch(conn, hits)
            pages_fetched  += 1
            total_ingested += inserted
            search_after    = [hits[-1]["sort"][0], hits[-1]["sort"][1]]

            save_checkpoint(total_ingested, search_after, pages_fetched)
            log.info("Página %s — %s inseridos | Total: %s",
                     pages_fetched, inserted, f"{total_ingested:,}")
            time.sleep(0.5)
    finally:
        conn.close()

    log.info("FIM — %s páginas, %s registros", pages_fetched, f"{total_ingested:,}")

run_ingestion()

2026-05-17 02:43:18,782 [INFO] Checkpoint encontrado: 1500 registros, retomando de search_after=[20240828000000, 1776402552478]
2026-05-17 02:43:18,784 [INFO] Iniciando. Já ingeridos: 1,500
C:\Users\fabri\AppData\Local\Temp\ipykernel_12204\580771055.py:17: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "updated_at": datetime.utcnow().isoformat(),
2026-05-17 02:43:29,482 [INFO] Página 4 — 500 inseridos | Total: 2,000
2026-05-17 02:43:45,832 [INFO] Página 5 — 500 inseridos | Total: 2,500
2026-05-17 02:43:56,261 [INFO] Página 6 — 500 inseridos | Total: 3,000
2026-05-17 02:44:07,796 [INFO] Página 7 — 500 inseridos | Total: 3,500
2026-05-17 02:44:16,945 [INFO] Página 8 — 500 inseridos | Total: 4,000
2026-05-17 02:44:26,800 [INFO] Página 9 — 500 inseridos | Total: 4,500
2026-05-17 02:44:36,340 [INFO] Página 10 — 500 inseridos | Total: 5

In [43]:
#Célula 9: extração do staging para analytics
SQL_EXTRACT = r"""
INSERT INTO analytics.processos (
    process_id, numero_cnj, tribunal, classe_codigo, classe_nome,
    data_ajuizamento, data_ultima_atualizacao,
    assunto_principal, assunto_codigo,
    grau, orgao_julgador, formato, tempo_tramitacao_dias
)
SELECT
    r.process_id,
    r.raw_json ->> 'numeroProcesso',
    r.tribunal,
    (r.raw_json -> 'classe' ->> 'codigo')::INTEGER,
    r.raw_json -> 'classe' ->> 'nome',
    TO_DATE(LEFT(r.raw_json ->> 'dataAjuizamento', 8), 'YYYYMMDD'),
    CASE
        WHEN r.raw_json ->> 'dataHoraUltimaAtualizacao' ~ '^\d{14}$'
        THEN TO_DATE(LEFT(r.raw_json ->> 'dataHoraUltimaAtualizacao', 8), 'YYYYMMDD')
        WHEN r.raw_json ->> 'dataHoraUltimaAtualizacao' ~ '^\d{4}-\d{2}-\d{2}'
        THEN LEFT(r.raw_json ->> 'dataHoraUltimaAtualizacao', 10)::DATE
        ELSE NULL
    END,
    CASE WHEN jsonb_array_length(r.raw_json -> 'assuntos') > 0
         THEN r.raw_json -> 'assuntos' -> 0 ->> 'nome' END,
    CASE WHEN jsonb_array_length(r.raw_json -> 'assuntos') > 0
         THEN (r.raw_json -> 'assuntos' -> 0 ->> 'codigo')::INTEGER END,
    r.raw_json ->> 'grau',
    r.raw_json -> 'orgaoJulgador' ->> 'nome',
    r.raw_json ->> 'formato',
    CASE
        WHEN r.raw_json ->> 'dataAjuizamento' IS NOT NULL
         AND r.raw_json ->> 'dataHoraUltimaAtualizacao' IS NOT NULL
        THEN (
            CASE
                WHEN r.raw_json ->> 'dataHoraUltimaAtualizacao' ~ '^\d{14}$'
                THEN TO_DATE(LEFT(r.raw_json ->> 'dataHoraUltimaAtualizacao', 8), 'YYYYMMDD')
                WHEN r.raw_json ->> 'dataHoraUltimaAtualizacao' ~ '^\d{4}-\d{2}-\d{2}'
                THEN LEFT(r.raw_json ->> 'dataHoraUltimaAtualizacao', 10)::DATE
                ELSE NULL
            END
            - TO_DATE(LEFT(r.raw_json ->> 'dataAjuizamento', 8), 'YYYYMMDD')
        )
        ELSE NULL
    END
FROM staging.decisoes_raw r
ON CONFLICT (process_id) DO UPDATE
    SET data_ultima_atualizacao = EXCLUDED.data_ultima_atualizacao,
        tempo_tramitacao_dias   = EXCLUDED.tempo_tramitacao_dias;
"""

with get_conn() as conn:
    with conn.cursor() as cur:
        cur.execute(SQL_EXTRACT)
        log.info("Extraídos para analytics.processos: %s linhas", cur.rowcount)
    conn.commit()

2026-05-17 02:58:58,077 [INFO] Extraídos para analytics.processos: 40232 linhas


In [48]:
#Célula 10: validação
queries = {
    "Volume staging":    "SELECT COUNT(*) AS total FROM staging.decisoes_raw",
    "Volume analytics":  "SELECT COUNT(*) AS total FROM analytics.processos",
    "Completude (%)": """
        SELECT
            ROUND(100.0 * COUNT(numero_cnj)        / COUNT(*), 1) AS pct_numero,
            ROUND(100.0 * COUNT(data_ajuizamento)  / COUNT(*), 1) AS pct_data,
            ROUND(100.0 * COUNT(assunto_principal) / COUNT(*), 1) AS pct_assunto,
            ROUND(100.0 * COUNT(orgao_julgador)    / COUNT(*), 1) AS pct_orgao
        FROM analytics.processos
    """,
    "Top 10 assuntos": """
        SELECT assunto_principal, COUNT(*) AS total
        FROM analytics.processos
        WHERE assunto_principal IS NOT NULL
        GROUP BY 1 ORDER BY 2 DESC LIMIT 10
    """,
    "Tramitação (percentis)": """
        SELECT
            PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY tempo_tramitacao_dias) AS p25,
            PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY tempo_tramitacao_dias) AS p50,
            PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY tempo_tramitacao_dias) AS p75,
            ROUND(AVG(tempo_tramitacao_dias), 0)                                AS media
        FROM analytics.processos WHERE tempo_tramitacao_dias > 0
    """,
}

with get_conn() as conn:
    for titulo, sql in queries.items():
        print(f"\n{'─'*50}\n {titulo}\n{'─'*50}")
        print(pd.read_sql(sql, conn).to_string(index=False))


──────────────────────────────────────────────────
 Volume staging
──────────────────────────────────────────────────


C:\Users\fabri\AppData\Local\Temp\ipykernel_12204\2371152282.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(pd.read_sql(sql, conn).to_string(index=False))
C:\Users\fabri\AppData\Local\Temp\ipykernel_12204\2371152282.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(pd.read_sql(sql, conn).to_string(index=False))
C:\Users\fabri\AppData\Local\Temp\ipykernel_12204\2371152282.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(pd.read_sql(sql, conn).to_string(index=False))
C:\Users\fabri\AppData\Local

 total
 40232

──────────────────────────────────────────────────
 Volume analytics
──────────────────────────────────────────────────
 total
 40232

──────────────────────────────────────────────────
 Completude (%)
──────────────────────────────────────────────────
 pct_numero  pct_data  pct_assunto  pct_orgao
      100.0     100.0         99.5      100.0

──────────────────────────────────────────────────
 Top 10 assuntos
──────────────────────────────────────────────────
                   assunto_principal  total
                  Verbas Rescisórias   4996
                   Rescisão Indireta   2340
                        Aviso Prévio   2146
          Adicional de Insalubridade   1958
Reconhecimento de Relação de Emprego   1660
           Adicional de Horas Extras   1562
         Multa do Artigo  477 da CLT   1296
                        Horas Extras   1018
                                FGTS    989
                Multa de 40% do FGTS    964

───────────────────────────────────